<a href="https://colab.research.google.com/drive/1YIcd9331iTscngkPy79OkgsFavBuuAuP#scrollTo=E1-bvxpBXLEV" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
!pip -q install -U transformers datasets evaluate accelerate rouge_score sentencepiece

In [ ]:
# Cell 2: Check the runtime environment

import os
import platform
import torch

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU detected. Please switch the runtime to GPU in Colab.")

In [ ]:
# Cell 3: Import libraries and set random seeds

import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Cell 4: Define file paths and experiment configuration

TRAIN_PATH = "/train.json"
VAL_PATH = "/val.json"
TEST_PATH = "/test.json"

# Cell 4

MODEL_CHECKPOINT = "google/pegasus-large"
OUTPUT_DIR = "./pegasus_large_dialogue_summarization"

MAX_SOURCE_LENGTH = 512      # 768 -> 512
MAX_TARGET_LENGTH = 48       # 64 -> 48

NUM_TRAIN_EPOCHS = 2         # 5 -> 2
LEARNING_RATE = 2e-5
NUM_BEAMS = 2                # 5 -> 2

PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4

# T4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

print("Model checkpoint:", MODEL_CHECKPOINT)
print("Output directory:", OUTPUT_DIR)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

In [ ]:
# Cell 5: Define helper functions for loading JSON files

def load_json_file(file_path: str) -> pd.DataFrame:
    """Load a JSON file that may be either a JSON array or JSON Lines."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.DataFrame([data])
        raise ValueError(f"Unsupported JSON structure in {file_path}")
    except json.JSONDecodeError:
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.DataFrame(records)

train_df = load_json_file(TRAIN_PATH)
val_df = load_json_file(VAL_PATH)
test_df = load_json_file(TEST_PATH)

print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("test_df shape:", test_df.shape)

In [ ]:
# Cell 6: Inspect the raw data

display(train_df.head(2))
print("Columns:", list(train_df.columns))

In [ ]:
# Cell 7: Keep only the required columns

required_columns = ["dialogue", "summary"]

train_df = train_df[required_columns].copy()
val_df = val_df[required_columns].copy()
test_df = test_df[required_columns].copy()

print("Remaining columns:", list(train_df.columns))

In [ ]:
# Cell 8: Clean and preprocess the text

def clean_dialogue_text(text: str) -> str:
    """Clean the dialogue text while preserving useful speaker information."""
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove placeholder tokens such as <file_gif>, <file_photo>, and similar markers.
    text = re.sub(r"<file_[^>]+>", " ", text)

    # Normalize whitespace.
    text = text.replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)

    return text.strip()

def clean_summary_text(text: str) -> str:
    """Clean the target summary text."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

for df in [train_df, val_df, test_df]:
    df["dialogue"] = df["dialogue"].apply(clean_dialogue_text)
    df["summary"] = df["summary"].apply(clean_summary_text)

    # Remove empty rows.
    df.dropna(subset=["dialogue", "summary"], inplace=True)
    df = df[(df["dialogue"].str.len() > 0) & (df["summary"].str.len() > 0)]

# Re-apply the empty-row filtering correctly after inplace operations.
train_df = train_df[(train_df["dialogue"].str.len() > 0) & (train_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)
val_df = val_df[(val_df["dialogue"].str.len() > 0) & (val_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)
test_df = test_df[(test_df["dialogue"].str.len() > 0) & (test_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)

print("train_df shape after cleaning:", train_df.shape)
print("val_df shape after cleaning:", val_df.shape)
print("test_df shape after cleaning:", test_df.shape)

display(train_df.head(2))

In [ ]:
# Cell 9: Load the tokenizer and inspect token length distributions

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def get_token_lengths(texts, tokenizer):
    return [len(tokenizer(text, truncation=False)["input_ids"]) for text in texts]

sample_size_for_stats = min(2000, len(train_df))
sample_dialogues = train_df["dialogue"].iloc[:sample_size_for_stats].tolist()
sample_summaries = train_df["summary"].iloc[:sample_size_for_stats].tolist()

dialogue_lengths = get_token_lengths(sample_dialogues, tokenizer)
summary_lengths = get_token_lengths(sample_summaries, tokenizer)

length_stats = pd.DataFrame(
    {
        "dialogue_tokens": pd.Series(dialogue_lengths),
        "summary_tokens": pd.Series(summary_lengths),
    }
)

display(length_stats.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("Current MAX_SOURCE_LENGTH:", MAX_SOURCE_LENGTH)
print("Current MAX_TARGET_LENGTH:", MAX_TARGET_LENGTH)

In [ ]:
# Cell 10: Convert pandas DataFrames to Hugging Face Dataset objects

dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)

dataset_dict

In [ ]:
# Cell 11: Define the tokenization function

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="Tokenizing datasets",
)

tokenized_datasets

In [ ]:
# 11-1: use a smaller subset for quick training

small_train_size = 3000
small_val_size = 500
small_test_size = 500

tokenized_datasets["train"] = tokenized_datasets["train"].select(
    range(min(small_train_size, len(tokenized_datasets["train"])))
)
tokenized_datasets["validation"] = tokenized_datasets["validation"].select(
    range(min(small_val_size, len(tokenized_datasets["validation"])))
)
tokenized_datasets["test"] = tokenized_datasets["test"].select(
    range(min(small_test_size, len(tokenized_datasets["test"])))
)

print(tokenized_datasets)

In [ ]:
# Cell 12: Load the model, data collator, and ROUGE metric

import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Some versions return a tuple, while others return a single ndarray.
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    labels = np.array(labels)

    # If predictions are logits instead of generated token ids,
    # convert them to token ids with argmax.
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Make sure the dtype is valid for decoding.
    predictions = predictions.astype(np.int64)

    # Replace invalid negative ids before decoding.
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)

    # Replace label masking id (-100) with the pad token id.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # ROUGE-LSum prefers newline-separated sentence boundaries.
    decoded_preds = ["\n".join(pred.split(". ")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.split(". ")) for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions
    ]
    result["gen_len"] = round(float(np.mean(prediction_lens)), 4)

    return result

In [ ]:
# New Cell: Load a fresh PEGASUS model before training

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Fresh model loaded from:", MODEL_CHECKPOINT)
print("Decoder start token id:", model.config.decoder_start_token_id)

In [ ]:
# Cell 13: Prepare mixed precision settings for Colab GPU

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

print("use_bf16:", use_bf16)
print("use_fp16:", use_fp16)

In [ ]:
# New Cell: Load a fresh PEGASUS model before training

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Fresh model loaded from:", MODEL_CHECKPOINT)
print("decoder_start_token_id:", model.config.decoder_start_token_id)

In [ ]:
# Cell 14: Define training arguments and trainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,
    optim="adafactor",
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("trainer created successfully")

In [ ]:
# Cell 15: Start full training

train_result = trainer.train()
train_result

In [ ]:
# Cell 16: Evaluate on the validation set with predict()

try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.pop_callback(NotebookProgressCallback)
except Exception:
    pass

validation_output = trainer.predict(tokenized_datasets["validation"])
validation_metrics = validation_output.metrics
validation_metrics

In [ ]:
# Cell 17: Evaluate on the test set with predict()

test_output = trainer.predict(tokenized_datasets["test"])
test_metrics = test_output.metrics
test_metrics

In [ ]:
# Cell 18: Save the final model and tokenizer

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model and tokenizer to:", OUTPUT_DIR)

In [ ]:
# Cell 19: Test the fine-tuned model on the provided sample dialogue

sample_dialogue = """Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye"""

sample_dialogue_clean = clean_dialogue_text(sample_dialogue)

inputs = tokenizer(
    sample_dialogue_clean,
    return_tensors="pt",
    max_length=MAX_SOURCE_LENGTH,
    truncation=True,
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

generated_ids = model.generate(
    **inputs,
    max_length=64,
    min_length=12,
    num_beams=5,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

generated_summary = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
)

print("Input dialogue:")
print(sample_dialogue_clean)
print("\nGenerated summary:")
print(generated_summary)

In [ ]:
# Cell: Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell: Define the Google Drive save path
DRIVE_SAVE_DIR = "/content/drive/MyDrive/LING 573/pegasus_large_dialogue_summarization_final"
print(DRIVE_SAVE_DIR)

In [ ]:
# Cell: Save the fine-tuned model and tokenizer to Google Drive
trainer.save_model(DRIVE_SAVE_DIR)
tokenizer.save_pretrained(DRIVE_SAVE_DIR)

print("Saved model and tokenizer to:", DRIVE_SAVE_DIR)

In [ ]:
# Cell: Reload the saved model and tokenizer from Google Drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_DIR = "/content/drive/MyDrive/LING 573/pegasus_large_dialogue_summarization_final"

reloaded_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
reloaded_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

print("Reloaded model from:", MODEL_DIR)

In [ ]:
# Cell: Login to the Hugging Face Hub

!pip -q install -U huggingface_hub

from huggingface_hub import login

login()

In [ ]:
# Cell 21: Push the model and tokenizer to the Hugging Face Hub

REPO_ID = "yunu919/pegasus-large-dialogue-summarization"

model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)

print("Uploaded to:", REPO_ID)

In [ ]:
# Optional Cell: Load the uploaded model from the Hugging Face Hub

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

REPO_ID = "yunu919/pegasus-large-dialogue-summarization"

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(REPO_ID)